# FieldMaps | Exploración y descarga interactiva de límites administrativos

Datos geoespaciales con Python, SQL y DuckDB

## Objetivo

Construir un procedimiento **reutilizable** para explorar una fuente geoespacial remota, identificar los valores administrativos disponibles y descargar los límites del territorio elegido **sin modificar manualmente las consultas SQL**.

**Secuencia de selección:**

1. Mostrar **todos los `adm0_src`** disponibles en la fuente y seleccionar uno con `input()`.
2. Mostrar los **`adm1_name`** del `adm0_src` elegido y seleccionar uno con `input()`.
3. Opcionalmente, elegir una unidad de **`adm3_name`** (se muestra junto con `adm2_name`) o conservar todos los registros de ADM1.
4. Recuperar las geometrías, validar el subconjunto, guardar un **GeoJSON** y mostrar un mapa interactivo.

La elección se puede introducir **por número o por texto exacto**, sin distinguir mayúsculas y minúsculas. Usar el número es especialmente útil si la fuente contiene tildes mal codificadas.

**Tecnologías:** DuckDB, SQL espacial, Pandas, GeoPandas y Lonboard. **Fuente:** [FieldMaps — `adm4_polygons.parquet`](https://data.fieldmaps.io/edge-matched/open/intl/adm4_polygons.parquet).

> **Importante:** `adm0_src`, `adm1_name`, `adm2_name` y `adm3_name` son campos del conjunto de datos. Sus significados concretos pueden variar entre países. El recurso se llama `adm4_polygons` y puede contener múltiples registros por unidad ADM3: filtrar por un nombre **no equivale a fusionar geometrías**.

## 1. Preparar el entorno

Se comprueba si el código se ejecuta en Google Colab o en Jupyter local. Si faltan paquetes, se instalan en el entorno del kernel actual. Es necesario disponer de **Internet** tanto para instalar paquetes faltantes como para consultar el archivo Parquet y, en Colab, descargar la extensión espacial.

La instalación automatizada es conveniente para reutilizar el notebook. Para entornos productivos, conviene fijar y registrar las versiones de las bibliotecas.

In [ ]:
import importlib.util
import os
import subprocess
import sys

environment = "colab" if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ else "local"
print(f"Entorno detectado: {environment}")

paquetes = {"duckdb": "duckdb", "geopandas": "geopandas", "pandas": "pandas", "lonboard": "lonboard"}
faltantes = [pip_nombre for modulo, pip_nombre in paquetes.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    print("Instalando dependencias faltantes:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
else:
    print("Dependencias principales disponibles.")

## 2. Cargar bibliotecas y definir utilidades de selección

La función `mostrar_opciones` muestra **todas** las alternativas con un índice. La función `seleccionar_opcion` valida la entrada y evita generar consultas con nombres que no existen. Una entrada numérica se interpreta como posición en la lista (desde 1); también puede introducirse un nombre tal como aparece impreso.

En ADM3, presionar **Enter** permite trabajar con **todo ADM1** sin seleccionar una unidad menor.

In [ ]:
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import geopandas as gpd
import pandas as pd
from IPython.display import display


def mostrar_opciones(opciones, titulo):
    """Imprime el listado completo y numerado, en el orden de la consulta."""
    print(f"\n{titulo} ({len(opciones)} opciones)")
    print("-" * 70)
    for n, opcion in enumerate(opciones, start=1):
        print(f"{n:>4}. {opcion}")


def seleccionar_opcion(opciones, titulo, permitir_todos=False):
    """Selecciona un elemento existente por número o texto, o None si corresponde."""
    if not opciones and not permitir_todos:
        raise ValueError(f"No hay valores disponibles para {titulo}.")

    while True:
        sufijo = " (Enter = todos)" if permitir_todos else ""
        entrada = input(f"Selecciona {titulo} por número o nombre{sufijo}: ").strip()
        if permitir_todos and entrada == "":
            return None
        if entrada.isdigit():
            posicion = int(entrada) - 1
            if 0 <= posicion < len(opciones):
                return opciones[posicion]
        coincidencias = [x for x in opciones if str(x).casefold() == entrada.casefold()]
        if len(coincidencias) == 1:
            return coincidencias[0]
        print("Opción no válida. Escribe un número del listado o copia el texto exacto.")


def nombre_archivo_seguro(texto):
    """Transforma un nombre administrativo en un fragmento de nombre de archivo."""
    sin_tildes = unicodedata.normalize("NFKD", str(texto))
    sin_tildes = "".join(c for c in sin_tildes if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", sin_tildes.casefold()).strip("_")[:65] or "sin_nombre"

## 3. Conectarse a DuckDB y declarar la fuente

DuckDB permite leer un Parquet desde una URL HTTP mediante `read_parquet(...)`. La extensión `spatial` permite recuperar las geometrías como **WKB** (`ST_AsWKB`), formato binario interoperable con GeoPandas.

El enlace se mantiene separado de los filtros; las entradas del usuario se envían como **parámetros SQL (`?`)** y no se concatenan directamente en la cláusula `WHERE`.

In [ ]:
parquet_url = "https://data.fieldmaps.io/edge-matched/open/intl/adm4_polygons.parquet"
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

# La URL es fija y controlada por el notebook; las selecciones se parametrizan después.
fuente_sql = f"read_parquet('{parquet_url}')"
fecha_consulta_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("Fuente:", parquet_url)
print("Consulta iniciada (UTC):", fecha_consulta_utc)

## 4. Examinar tamaño y estructura del conjunto de datos

Se cuenta el número de filas y se revisan las columnas antes de seleccionar un país. El total corresponde a **registros del archivo completo**, no a países ni a comunas; puede cambiar cuando el proveedor actualice la fuente.

La geometría se almacena en el campo `geometry`. Para la selección territorial se usan `adm0_src`, `adm1_name`, `adm2_name` y `adm3_name`.

In [ ]:
total_registros = con.execute(f"SELECT COUNT(*) FROM {fuente_sql}").fetchone()[0]
esquema = con.execute(f"DESCRIBE SELECT * FROM {fuente_sql}").df()
print(f"Total de registros de la fuente: {total_registros:,}")
display(esquema)

## 5. Mostrar **todos** los valores de `adm0_src`

Esta primera exploración solicita los valores distintos de `adm0_src` disponibles en el Parquet, excluyendo nulos y cadenas vacías. **La lista se imprime completa y se numera** para poder revisarla antes de introducir cualquier selección.

> En este conjunto de datos `adm0_src` es el campo utilizado como filtro de país o fuente administrativa de nivel cero. Conviene conservar sus valores originales, sin reemplazarlos por nombres supuestos.

In [ ]:
opciones_adm0 = [fila[0] for fila in con.execute(f"""
    SELECT DISTINCT adm0_src
    FROM {fuente_sql}
    WHERE adm0_src IS NOT NULL AND TRIM(adm0_src) <> ''
    ORDER BY adm0_src
""").fetchall()]

mostrar_opciones(opciones_adm0, "Valores disponibles de adm0_src")

## 6. Seleccionar `adm0_src` mediante `input()`

Ahora se introduce el **número** mostrado en el paso anterior o el texto exacto del valor. El resultado se guarda en `adm0_src_seleccionado` y se utiliza en las consultas posteriores.

**Ejemplo de uso:** si `CHL` aparece en la lista, puede escribirse `CHL` o su número de posición. Ningún país queda fijado en el código.

In [ ]:
adm0_src_seleccionado = seleccionar_opcion(opciones_adm0, "adm0_src")
print("ADM0 seleccionado:", adm0_src_seleccionado)

## 7. Mostrar los `adm1_name` asociados al ADM0 seleccionado

La segunda consulta aplica un filtro SQL con el valor elegido en el paso anterior y devuelve los nombres ADM1 existentes **solo para ese ADM0**. Se evitan nombres de región codificados manualmente, por lo que el flujo es adaptable a distintos territorios.

In [ ]:
opciones_adm1 = [fila[0] for fila in con.execute(f"""
    SELECT DISTINCT adm1_name
    FROM {fuente_sql}
    WHERE adm0_src = ?
      AND adm1_name IS NOT NULL
      AND TRIM(adm1_name) <> ''
    ORDER BY adm1_name
""", [adm0_src_seleccionado]).fetchall()]

mostrar_opciones(opciones_adm1, f"Valores adm1_name para {adm0_src_seleccionado}")
if not opciones_adm1:
    raise ValueError("El ADM0 elegido no contiene ADM1 con nombre. Vuelve al paso 6 y cambia la selección.")

## 8. Seleccionar `adm1_name` mediante `input()`

La selección por índice evita tener que copiar cadenas extensas o que presenten errores de codificación, como ocurría con ciertos nombres en el ejemplo original del Biobío. El filtro conserva el texto **exacto** almacenado en los datos.

In [ ]:
adm1_name_seleccionado = seleccionar_opcion(opciones_adm1, "adm1_name")
print("ADM1 seleccionado:", adm1_name_seleccionado)

## 9. Opcional: inspeccionar ADM3 dentro de ADM1 y elegir una unidad

Se listan las combinaciones distintas de `adm2_name` y `adm3_name`, porque un mismo nombre ADM3 podría repetirse en unidades ADM2 diferentes. Hay dos posibilidades:

- **Presionar Enter:** recuperar todos los registros asociados al ADM1 seleccionado.
- **Ingresar número o texto:** recuperar solo los registros de la combinación ADM2 + ADM3 elegida.

Si no existen nombres ADM3 para el territorio, el notebook continúa con todo ADM1. La nomenclatura de estos niveles **no siempre equivale a provincia y comuna** fuera del contexto chileno.

In [ ]:
pares_adm3 = con.execute(f"""
    SELECT DISTINCT adm2_name, adm3_name
    FROM {fuente_sql}
    WHERE adm0_src = ? AND adm1_name = ?
      AND adm3_name IS NOT NULL AND TRIM(adm3_name) <> ''
    ORDER BY adm2_name NULLS FIRST, adm3_name
""", [adm0_src_seleccionado, adm1_name_seleccionado]).fetchall()

etiquetas_adm3 = [
    f"{adm3} | ADM2: {adm2 if adm2 is not None else '(sin nombre)'}"
    for adm2, adm3 in pares_adm3
]

mostrar_opciones(etiquetas_adm3, "Unidades ADM3 disponibles (selección opcional)")
eleccion_adm3 = seleccionar_opcion(etiquetas_adm3, "ADM3", permitir_todos=True)

if eleccion_adm3 is None:
    adm2_name_seleccionado = None
    adm3_name_seleccionado = None
    print("Sin filtro ADM3: se extraerá todo el ADM1 seleccionado.")
else:
    indice_adm3 = etiquetas_adm3.index(eleccion_adm3)
    adm2_name_seleccionado, adm3_name_seleccionado = pares_adm3[indice_adm3]
    print("ADM2 seleccionado:", adm2_name_seleccionado)
    print("ADM3 seleccionado:", adm3_name_seleccionado)

## 10. Extraer únicamente las geometrías seleccionadas

Se construye una consulta parametrizada: `adm0_src` y `adm1_name` son obligatorios, mientras que ADM2 + ADM3 son opcionales. Se solicitan solo los campos que necesita el análisis. La geometría espacial se convierte en WKB con `ST_AsWKB(geometry)`.

**Por qué parametrizar:** el uso de `?` mantiene las entradas del usuario separadas del código SQL, permite trabajar con apostrofes y caracteres especiales y evita errores por concatenación de cadenas.

In [ ]:
sql_geometrias = f"""
    SELECT adm0_src, adm1_name, adm2_name, adm3_name,
           ST_AsWKB(geometry) AS geometry_wkb
    FROM {fuente_sql}
    WHERE adm0_src = ? AND adm1_name = ?
"""
parametros = [adm0_src_seleccionado, adm1_name_seleccionado]

if adm3_name_seleccionado is not None:
    sql_geometrias += " AND adm3_name = ? AND adm2_name IS NOT DISTINCT FROM ?"
    parametros.extend([adm3_name_seleccionado, adm2_name_seleccionado])

tabla_seleccionada = con.execute(sql_geometrias, parametros).df()
if tabla_seleccionada.empty:
    raise ValueError("La consulta no devolvió registros. Revisa los valores seleccionados.")
print(f"Registros recuperados: {len(tabla_seleccionada):,}")
display(tabla_seleccionada.drop(columns=["geometry_wkb"]).head(10))

## 11. Crear un GeoDataFrame y realizar controles básicos

`GeoPandas` interpreta el WKB como geometría y vincula cada polígono con sus atributos territoriales. Se asigna **EPSG:4326** siguiendo la suposición del notebook original; **asignar un CRS no reproyecta ni confirma el CRS de la fuente**.

También se informan las geometrías nulas y las que no pasan la comprobación de validez de GeoPandas. Estos controles ayudan a diferenciar un problema de calidad de datos de uno de selección.

In [ ]:
wkb = tabla_seleccionada["geometry_wkb"].map(
    lambda geometria: bytes(geometria) if geometria is not None else None
)

geometrias = gpd.GeoSeries.from_wkb(wkb, crs="EPSG:4326")
gdf = gpd.GeoDataFrame(
    tabla_seleccionada.drop(columns=["geometry_wkb"]),
    geometry=geometrias,
    crs="EPSG:4326",
)

n_nulas = int(gdf.geometry.isna().sum())
no_nulas = gdf.geometry.dropna()
n_invalidas = int((~no_nulas.is_valid).sum())

print(f"Registros: {len(gdf):,}")
print(f"Geometrías nulas: {n_nulas:,} | Geometrías no válidas: {n_invalidas:,}")
print("CRS declarado:", gdf.crs)
display(gdf.head(10))

## 12. Resumir los registros por ADM2

Un resumen por `adm2_name` permite comprobar la distribución de **filas o geometrías** en el subconjunto. No deben interpretarse automáticamente como el número de comunas o de unidades administrativas únicas: el recurso de origen contiene polígonos ADM4 y puede haber varias filas por unidad ADM3.

In [ ]:
resumen_adm2 = (
    gdf.groupby("adm2_name", dropna=False)
       .size()
       .rename("n_registros")
       .reset_index()
       .sort_values("n_registros", ascending=False)
)
display(resumen_adm2)

## 13. Exportar GeoJSON a una carpeta local o a Google Drive

Se ofrece una pregunta final para elegir el destino. La opción predeterminada es una carpeta **`salidas_geojson`** en el directorio actual, compatible tanto con Jupyter local como con Colab. Si se responde `s` en Colab, se intenta montar **Google Drive** y guardar el resultado en `MyDrive/FieldMaps_geojson`.

El nombre del archivo se construye desde ADM0, ADM1 y, si corresponde, ADM3. Se excluyen únicamente las geometrías nulas del archivo exportado y se informa cuántas fueron omitidas. Los registros no se disuelven ni fusionan.

In [ ]:
carpeta_salida = Path.cwd() / "salidas_geojson"

respuesta_drive = input("¿Guardar en Google Drive? (s/N; solo en Colab): ").strip().casefold()
if respuesta_drive in ("s", "si", "sí"):
    if environment == "colab":
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            carpeta_salida = Path("/content/drive/MyDrive/FieldMaps_geojson")
        except Exception as exc:
            print(f"No se pudo montar Google Drive ({exc}). Se usará la carpeta local.")
    else:
        print("Google Drive no se monta automáticamente fuera de Colab. Se usará la carpeta local.")

carpeta_salida.mkdir(parents=True, exist_ok=True)
partes_nombre = [adm0_src_seleccionado, adm1_name_seleccionado]
partes_nombre.append(adm3_name_seleccionado if adm3_name_seleccionado is not None else "adm1_completo")
nombre_geojson = "_".join(nombre_archivo_seguro(p) for p in partes_nombre) + ".geojson"
ruta_geojson = carpeta_salida / nombre_geojson

exportable_gdf = gdf.loc[gdf.geometry.notna()].copy()
if exportable_gdf.empty:
    raise ValueError("No hay geometrías no nulas para exportar.")

exportable_gdf.to_file(ruta_geojson, driver="GeoJSON", encoding="utf-8")
print("GeoJSON generado:", ruta_geojson.resolve())
print(f"Registros exportados: {len(exportable_gdf):,}")
if n_nulas:
    print(f"Advertencia: se omitieron {n_nulas:,} registros con geometría nula.")

## 14. Visualizar las geometrías seleccionadas

`Lonboard` crea un mapa interactivo desde el `GeoDataFrame`. Este paso no modifica el GeoJSON: si el visor no está disponible en el entorno, el archivo ya exportado sigue siendo válido.

La visualización es exploratoria. Para realizar mediciones de área o distancia se necesitaría comprobar el CRS de origen y emplear una proyección métrica adecuada.

In [ ]:
try:
    import lonboard
    mapa = lonboard.viz(exportable_gdf)
    display(mapa)
except Exception as exc:
    print(f"No se pudo mostrar el mapa con Lonboard: {exc}")
    print("El GeoJSON exportado sigue disponible en:", ruta_geojson)

---

## Resultados y competencias demostradas

Este notebook permite **repetir el mismo procedimiento para distintas combinaciones de ADM0, ADM1 y ADM3** sin editar las consultas. Sus principales entregables son el listado de valores disponibles, el subconjunto geoespacial, un resumen de registros, un **GeoJSON** y una visualización interactiva.

### Limitaciones y posibles mejoras

- **Cobertura y vigencia:** los nombres y polígonos dependen de la versión de FieldMaps consultada. Revisar metadatos, fecha, licencia y fuentes antes de usarlos para decisiones técnicas.
- **Jerarquías ADM:** `adm4_polygons` puede contener varios polígonos por unidad ADM3. Si se requiere **un único límite agregado**, estudiar una disolución (`dissolve`) por el identificador territorial adecuado.
- **CRS:** la asignación `EPSG:4326` reproduce el supuesto del ejemplo original y debe verificarse con metadatos antes de análisis métricos.
- **Codificación:** algunos nombres pueden presentar caracteres alterados; elegir por índice evita tener que teclearlos, pero no corrige la fuente.
- **Rendimiento:** una selección administrativa muy extensa puede generar un GeoJSON grande o una visualización lenta. Se recomienda extraer territorios acotados.
- **Ejecución:** las celdas interactivas requieren responder a `input()`; al cambiar ADM0 o ADM1, ejecutar nuevamente **todas las celdas posteriores**, no solo la exportación.

**Fuente de datos:** [FieldMaps — ADM4 polygons](https://data.fieldmaps.io/edge-matched/open/intl/adm4_polygons.parquet). No se incluyen cifras ni mapas almacenados de una ejecución anterior para evitar asociarlos a selecciones nuevas.